# Train HMM, RNN & VAE on Symbolic Melodies

1. Load a melody dataset via `src/datasets.py` — set `DATASET` below to
   `"jsb"` (Bach chorales), `"nottingham"` (folk tunes), or `"weimar"`
   (jazz solos).
2. Label-encode MIDI pitches into integer symbols.
3. Train a `BasicHMM`, an `RNNModel`, and a `VAEModel` (all three implement
   the `SequenceModel` interface).
4. Generate a melody from each model.
5. Render each melody to a WAV using real piano samples.
6. Bonus: interpolate between two melodies in the VAE's latent space.

Project layout: input datasets live in `data/`, code in `src/`, and every
generated WAV/PNG in `results/`.

For a VAE-only driver with the dataset toggleable from the command line, see
`src/vae_music.py` (e.g. `python src/vae_music.py --dataset weimar`).

In [1]:
import os
import sys

import numpy as np

try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    HERE = os.getcwd()
sys.path.insert(0, os.path.join(HERE, "src"))

from basic_hmm import BasicHMM
from rnn_model import RNNModel
from vae_model import VAEModel
import datasets
import render

In [2]:
# Configuration
DATA_DIR = os.path.join(HERE, "data")
RESULTS_DIR = os.path.join(HERE, "results")
SAMPLE_DIR = os.path.join(DATA_DIR, "piano_samples")

# Which dataset to train on -- see datasets.py for the full registry.
# Switch back to the Bach chorales with DATASET = "jsb", or to folk tunes
# with DATASET = "nottingham".
DATASET = "weimar"
DATA_PATHS = {
    "jsb": os.path.join(DATA_DIR, "jsb-chorales-16th.pkl"),
    "nottingham": os.path.join(DATA_DIR, "nottingham_midi"),
    "weimar": os.path.join(DATA_DIR, "wjazzd.db"),
}

SEED = 42
N_NOTES = 512
OUTPUTS = {
    "hmm": os.path.join(RESULTS_DIR, "hmm_melody_weimar.wav"),
    "rnn": os.path.join(RESULTS_DIR, "rnn_melody_weimar.wav"),
    "vae": os.path.join(RESULTS_DIR, "vae_melody_weimar.wav"),
}

In [3]:
# Load data and encode melodies
melodies = datasets.load_melodies(DATASET, path=DATA_PATHS[DATASET])
sequences, encoder = datasets.encode_melodies(melodies)
all_notes = np.concatenate(melodies)
print(f"{datasets.DATASETS[DATASET]['name']}: {len(melodies)} melodies  "
      f"notes: {len(all_notes)}  symbols: {len(encoder.classes_)}  range: "
      f"{all_notes.min()}-{all_notes.max()} (MIDI)")

Weimar Jazz Database (jazz solos): 456 melodies  notes: 200809  symbols: 62  range: 36-97 (MIDI)

In [4]:
# Shared helper: train a model and generate a melody
def run_model(name, model):
    print(f"\n=== {name} ===")
    model.fit(sequences)
    symbols = model.generate(N_NOTES, seed=SEED)
    midi = encoder.inverse_transform(symbols).astype(int)
    print(f"Generated melody (MIDI): {list(midi)}")
    return midi

## Tonal bias (HMM only)

We bias the HMM's emission probabilities toward the notes of a chosen key, so
the generated melody is more "tonally correct".

**Key choice:** rather than hardcoding a key (the right choice differs by
dataset — Bach chorales sit in sharp keys, Nottingham's tunes lean G/D/A
major, Weimar jazz solos roam much further), we auto-detect the tonic by
scoring every major-key pitch-class set against the loaded melodies'
pitch-class histogram and keeping the best match.

`music21` is the canonical music-theory library (it has `Key`, `Scale`, key
detection, and even a Bach chorale corpus), but it isn't installed here — and
a key is just a tonic plus an interval pattern, so we compute it directly.

In [5]:
# --- Tonal bias for the HMM -------------------------------------------
# Bias the HMM's emission probabilities toward notes in the dataset's
# best-fit major key, so the generated melody is more "tonally correct". A
# key is just a tonic pitch class plus an interval pattern (no music-theory
# library needed); we score every tonic against the dataset's own
# pitch-class histogram and keep the best match.
MAJOR = [0, 2, 4, 5, 7, 9, 11]          # semitone offsets (major scale)
PITCH_NAMES = ["C", "Db", "D", "Eb", "E", "F", "Gb", "G", "Ab", "A", "Bb", "B"]
SCALE_BOOST = 4.0    # how strongly to favour in-scale notes over chromatic ones


def scale_pitch_classes(tonic):
    return {(tonic + i) % 12 for i in MAJOR}


def best_fit_tonic(notes):
    """Major-key tonic covering the most weight in `notes`'s pitch-class
    histogram (a simple stand-in for real key-finding algorithms such as
    Krumhansl-Schmuckler)."""
    counts = np.bincount(np.asarray(notes) % 12, minlength=12)
    scores = [sum(counts[pc] for pc in scale_pitch_classes(t)) for t in range(12)]
    return int(np.argmax(scores))


KEY_TONIC = best_fit_tonic(all_notes)
in_scale = scale_pitch_classes(KEY_TONIC)
emission_weights = np.array(
    [SCALE_BOOST if (int(m) % 12) in in_scale else 1.0
     for m in encoder.classes_],
    dtype=float,
)
print(f"Best-fit key: {PITCH_NAMES[KEY_TONIC]} major  pitch classes = "
      f"{sorted(in_scale)}")
print(f"In-scale symbols weighted {SCALE_BOOST}x higher at sampling time")

Best-fit key: Bb major  pitch classes = [0, 2, 3, 5, 7, 9, 10]

In-scale symbols weighted 4.0x higher at sampling time

In [6]:
# Model 1: BasicHMM (with tonal bias toward the dataset's best-fit key)
hmm = BasicHMM(n_components=8, n_iter=50, random_state=SEED,
               emission_weights=emission_weights)
hmm_melody = run_model("BasicHMM", hmm)


=== BasicHMM ===

Generated melody (MIDI): [np.int64(78), np.int64(82), np.int64(87), np.int64(82), np.int64(79), np.int64(85), np.int64(83), np.int64(79), np.int64(77), np.int64(81), np.int64(77), np.int64(84), np.int64(77), np.int64(75), np.int64(81), np.int64(87), np.int64(77), np.int64(79), np.int64(79), np.int64(75), np.int64(76), np.int64(77), np.int64(82), np.int64(77), np.int64(78), np.int64(74), np.int64(82), np.int64(82), np.int64(80), np.int64(77), np.int64(79), np.int64(82), np.int64(80), np.int64(77), np.int64(74), np.int64(74), np.int64(69), np.int64(67), np.int64(65), np.int64(65), np.int64(63), np.int64(62), np.int64(67), np.int64(70), np.int64(67), np.int64(65), np.int64(63), np.int64(65), np.int64(60), np.int64(62), np.int64(65), np.int64(67), np.int64(65), np.int64(62), np.int64(70), np.int64(72), np.int64(74), np.int64(76), np.int64(70), np.int64(65), np.int64(65), np.int64(67), np.int64(63), np.int64(61), np.int64(69), np.int64(67), np.int64(63), np.int64(58), np.int64(74), np.int64

In [7]:
# Model 2: RNN
rnn = RNNModel(embed_dim=32, hidden_size=64, num_layers=1, context_len=32,
               epochs=100, batch_size=64, lr=1e-3, temperature=1.0,
               random_state=SEED)
rnn_melody = run_model("RNN", rnn)


=== RNN ===

  RNN epoch 1/100  loss 3.3497

  RNN epoch 20/100  loss 2.3181

  RNN epoch 40/100  loss 2.2641

  RNN epoch 60/100  loss 2.2329

  RNN epoch 80/100  loss 2.2109

  RNN epoch 100/100  loss 2.1936

Generated melody (MIDI): [np.int64(60), np.int64(61), np.int64(60), np.int64(58), np.int64(55), np.int64(58), np.int64(63), np.int64(61), np.int64(63), np.int64(65), np.int64(60), np.int64(65), np.int64(69), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(69), np.int64(72), np.int64(72), np.int64(72), np.int64(72), np.int64(68), np.int64(65), np.int64(64), np.int64(63), np.int64(65), np.int64(67), np.int64(65), np.int64(68), np.int64(65), np.int64(67), np.int64(68), np.int64(72), np.int64(77), np.int64(72), np.int64(70), np.int64(65), np.int64(70), np.int64(65), np.int64(70), np.int64(69), np.int64(65), np.int64(65), np.int64(67), np.int64(65), np.int64(62), np.int64(70), np.int64(55), np.int64(56), np.int64(59), np.int64(55), np.int64(63), np.int64(59), np.int64(62), np.int64(60), np.int64(61), np.int64(63), np.int64(65), np.int64(66), np.int64(68), np.int64(70), np.int64(72), np.int64(73), np.int64(72), np.int64(71), np.int64(70), np.int64

In [8]:
# Model 3: VAE
vae = VAEModel(embed_dim=32, hidden_size=64, latent_dim=8, context_len=32,
               epochs=100, batch_size=64, lr=1e-3, beta=0.5, temperature=1.0,
               random_state=SEED)
vae_melody = run_model("VAE", vae)


=== VAE ===

  VAE epoch 1/100  recon 101.8569  kl 6.0220  beta 0.010

  VAE epoch 20/100  recon 69.6850  kl 3.8828  beta 0.200

  VAE epoch 40/100  recon 66.6124  kl 5.0557  beta 0.400

  VAE epoch 60/100  recon 65.2869  kl 5.4176  beta 0.500

  VAE epoch 80/100  recon 64.0725  kl 6.1467  beta 0.500

  VAE epoch 100/100  recon 63.1196  kl 6.7301  beta 0.500

Generated melody (MIDI): [np.int64(60), np.int64(61), np.int64(60), np.int64(58), np.int64(57), np.int64(58), np.int64(72), np.int64(73), np.int64(74), np.int64(73), np.int64(74), np.int64(72), np.int64(70), np.int64(67), np.int64(69), np.int64(70), np.int64(69), np.int64(68), np.int64(67), np.int64(65), np.int64(63), np.int64(62), np.int64(57), np.int64(55), np.int64(53), np.int64(52), np.int64(58), np.int64(63), np.int64(64), np.int64(61), np.int64(65), np.int64(62), np.int64(63), np.int64(65), np.int64(66), np.int64(67), np.int64(66), np.int64(64), np.int64(65), np.int64(64), np.int64(62), np.int64(60), np.int64(59), np.int64(62), np.int64(65), np.int64(66), np.int64(64), np.int64(65), np.int64(66), np.int64(70), np.int64(66), np.int64(62), np.int64(59), np.int64(65), np.int64(63), np.int64(59), np.int64(62), np.int64(60), np.int64(61), np.int64(60), np.int64(59), np.int64(60), np.int64(61), np.int64(60), np.int64(59), np.int64(62), np.int64(60), np.int64(59), np.int64(60), np.int64

In [9]:
import importlib

# Render all three melodies to WAV with real piano samples
importlib.reload(render)

for name, melody in (("hmm", hmm_melody), ("rnn", rnn_melody), ("vae", vae_melody)):
    path = render.render_melody(melody, OUTPUTS[name], SAMPLE_DIR)
    print(f"Wrote {path}")

Wrote /Users/fabian/PycharmProjects/Autoencoder/results/hmm_melody_weimar.wav

Wrote /Users/fabian/PycharmProjects/Autoencoder/results/rnn_melody_weimar.wav

Wrote /Users/fabian/PycharmProjects/Autoencoder/results/vae_melody_weimar.wav

## Bonus: latent-space interpolation (VAE only)

The HMM and RNN only support autoregressive sampling. The VAE's latent
bottleneck also lets us *encode* two real melodies to Gaussian means, walk a
straight line between them in latent space, and decode each waypoint — the
same latent-traversal idea as `inspect_latent_point` in the MNIST VAE
notebook, applied to music instead of pixels.

In [10]:
# Encode two melodies, linearly interpolate their latent means, and decode
# each waypoint into a melody of its own.
seq_a, seq_b = sequences[0], sequences[1]
mu_a, _ = vae.encode(seq_a)
mu_b, _ = vae.encode(seq_b)

N_STEPS = 5
N_INTERP_NOTES = 64
interp_melodies = []
for step in range(N_STEPS):
    t = step / (N_STEPS - 1)
    z = (1 - t) * mu_a + t * mu_b
    symbols = vae.decode(z, N_INTERP_NOTES, seed=SEED)
    interp_melodies.append(encoder.inverse_transform(symbols).astype(int))
    print(f"t={t:.2f}: {list(interp_melodies[-1])}")

interp_path = os.path.join(RESULTS_DIR, "vae_interpolation.wav")
path = render.render_melody(
    [n for m in interp_melodies for n in m], interp_path, SAMPLE_DIR
)
print(f"Wrote {path}")

t=0.00: [np.int64(60), np.int64(61), np.int64(63), np.int64(58), np.int64(81), np.int64(73), np.int64(72), np.int64(75), np.int64(74), np.int64(73), np.int64(74), np.int64(72), np.int64(70), np.int64(67), np.int64(70), np.int64(69), np.int64(67), np.int64(70), np.int64(67), np.int64(69), np.int64(72), np.int64(79), np.int64(84), np.int64(86), np.int64(82), np.int64(85), np.int64(88), np.int64(82), np.int64(81), np.int64(80), np.int64(78), np.int64(72), np.int64(75), np.int64(74), np.int64(72), np.int64(72), np.int64(77), np.int64(72), np.int64(70), np.int64(65), np.int64(62), np.int64(65), np.int64(70), np.int64(69), np.int64(67), np.int64(65), np.int64(64), np.int64(65), np.int64(67), np.int64(70), np.int64(67), np.int64(69), np.int64(70), np.int64(65), np.int64(63), np.int64(59), np.int64(62), np.int64(60), np.int64(58), np.int64(60), np.int64(62), np.int64(65), np.int64(67), np.int64(70)]

t=0.25: [np.int64(60), np.int64(61), np.int64(63), np.int64(58), np.int64(81), np.int64(73), np.int64(72), np.int64(75), np.int64(74), np.int64(73), np.int64(74), np.int64(72), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(72), np.int64(70), np.int64(70), np.int64(69), np.int64(72), np.int64(72), np.int64(67), np.int64(69), np.int64(70), np.int64(64), np.int64(67), np.int64(63), np.int64(64), np.int64(60), np.int64(65), np.int64(67), np.int64(63), np.int64(65), np.int64(67), np.int64(72), np.int64(71), np.int64(72), np.int64(70), np.int64(65), np.int64(62), np.int64(60), np.int64(59), np.int64(62), np.int64(67), np.int64(66), np.int64(67), np.int64(65), np.int64(66), np.int64(62), np.int64(65), np.int64(62), np.int64(59), np.int64(55), np.int64(63), np.int64(59), np.int64(62), np.int64(60), np.int64(61), np.int64(60), np.int64(55), np.int64(60), np.int64(63), np.int64(60)]

t=0.50: [np.int64(73), np.int64(74), np.int64(73), np.int64(74), np.int64(73), np.int64(73), np.int64(72), np.int64(73), np.int64(68), np.int64(73), np.int64(73), np.int64(72), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(69), np.int64(72), np.int64(72), np.int64(72), np.int64(73), np.int64(74), np.int64(73), np.int64(72), np.int64(73), np.int64(68), np.int64(72), np.int64(73), np.int64(72), np.int64(68), np.int64(67), np.int64(68), np.int64(72), np.int64(77), np.int64(72), np.int64(70), np.int64(65), np.int64(62), np.int64(65), np.int64(70), np.int64(72), np.int64(73), np.int64(74), np.int64(72), np.int64(73), np.int64(72), np.int64(70), np.int64(67), np.int64(68), np.int64(70), np.int64(65), np.int64(63), np.int64(62), np.int64(60), np.int64(60), np.int64(60), np.int64(60), np.int64(65), np.int64(60), np.int64(60), np.int64(60)]

t=0.75: [np.int64(73), np.int64(71), np.int64(70), np.int64(70), np.int64(73), np.int64(73), np.int64(72), np.int64(73), np.int64(68), np.int64(73), np.int64(73), np.int64(72), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(69), np.int64(72), np.int64(72), np.int64(67), np.int64(68), np.int64(70), np.int64(68), np.int64(67), np.int64(63), np.int64(65), np.int64(67), np.int64(65), np.int64(63), np.int64(60), np.int64(65), np.int64(67), np.int64(72), np.int64(65), np.int64(67), np.int64(70), np.int64(65), np.int64(62), np.int64(65), np.int64(70), np.int64(69), np.int64(67), np.int64(65), np.int64(64), np.int64(65), np.int64(62), np.int64(58), np.int64(55), np.int64(53), np.int64(55), np.int64(55), np.int64(58), np.int64(59), np.int64(62), np.int64(60), np.int64(61), np.int64(60), np.int64(65), np.int64(60), np.int64(60), np.int64(60)]

t=1.00: [np.int64(73), np.int64(71), np.int64(70), np.int64(70), np.int64(73), np.int64(73), np.int64(72), np.int64(73), np.int64(68), np.int64(73), np.int64(73), np.int64(72), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(70), np.int64(72), np.int64(72), np.int64(72), np.int64(73), np.int64(70), np.int64(68), np.int64(67), np.int64(63), np.int64(65), np.int64(67), np.int64(65), np.int64(63), np.int64(63), np.int64(65), np.int64(66), np.int64(63), np.int64(65), np.int64(63), np.int64(65), np.int64(65), np.int64(65), np.int64(65), np.int64(65), np.int64(64), np.int64(65), np.int64(66), np.int64(64), np.int64(65), np.int64(63), np.int64(58), np.int64(55), np.int64(56), np.int64(53), np.int64(65), np.int64(63), np.int64(63), np.int64(65), np.int64(63), np.int64(61), np.int64(60), np.int64(65), np.int64(60), np.int64(63), np.int64(60)]

Wrote /Users/fabian/PycharmProjects/Autoencoder/results/vae_interpolation.wav